In [ ]:
# Sonali vishal pawar
# Task5_POS_Tagging_Chunking

In [ ]:
# Project Title

## Fine-Tuning DistilBERT for POS Tagging and Chunking using CoNLL-2003 Dataset

In [ ]:
# Project Summary
## This project focuses on fine-tuning the DistilBERT transformer model for token classification tasks such as Part-of-Speech (POS) Tagging and Chunking using the CoNLL-2003 dataset. The objective of the project is to identify grammatical tags and phrase-level structures in sentences using Natural Language Processing (NLP) techniques.
The project includes data preprocessing, tokenization, label alignment, model training, evaluation, and inference. The DistilBERT model was trained using Hugging Face Transformers and evaluated using Precision, Recall, and F1 Score.
The project demonstrates the effectiveness of transformer-based models for sequence labeling tasks in NLP.

In [ ]:
# Objective
To understand token classification using transformer models.
To perform POS tagging and chunking using DistilBERT.
To tokenize text data and align labels correctly.
To fine-tune a pre-trained transformer model.
To evaluate sequence labeling performance using standard metrics.
To compare POS tagging and chunking tasks.

In [ ]:
# Problem Statement

Traditional machine learning models often struggle to understand contextual relationships between words in Natural Language Processing tasks such as POS tagging and chunking.

The goal of this project is to build an efficient token classification system using DistilBERT that can accurately assign grammatical tags and identify phrase structures in sentences.

The project aims to improve sequence labeling performance using transformer-based deep learning models.

In [ ]:
# Install Libraries

In [ ]:
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Import Libraries

In [ ]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
# Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "tomaarsen/conll2003"
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/316k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/288k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [ ]:
# Reduce Dataset Size (CPU Friendly)

In [ ]:
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))

small_val_dataset = dataset["validation"].shuffle(seed=42).select(range(500))

In [ ]:
label_names = dataset["train"].features["ner_tags"].feature.names

print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [ ]:
# Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [ ]:
# Apply Tokenization

In [ ]:
tokenized_train = small_train_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_val = small_val_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# Load Model

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names)
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Data Collator

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
# Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1
)

In [ ]:
# Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

In [ ]:
# Train Model

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.307316


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.30731634521484374, metrics={'train_runtime': 660.7148, 'train_samples_per_second': 3.027, 'train_steps_per_second': 0.757, 'total_flos': 17807821413984.0, 'train_loss': 0.30731634521484374, 'epoch': 1.0})

In [ ]:
# Prediction

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_val)

predictions = np.argmax(predictions, axis=2)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Convert Predictions

In [ ]:
true_predictions = [
    [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

true_labels = [
    [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

In [ ]:
# Evaluation Metrics

In [ ]:
from seqeval.metrics import precision_score
from seqeval.metrics import recall_score
from seqeval.metrics import f1_score

print("Precision:", precision_score(true_labels, true_predictions))

print("Recall:", recall_score(true_labels, true_predictions))

print("F1 Score:", f1_score(true_labels, true_predictions))

Precision: 0.7851063829787234
Recall: 0.827354260089686
F1 Score: 0.8056768558951964


In [ ]:
# Classification Report

In [ ]:
from seqeval.metrics import classification_report

print(classification_report(true_labels, true_predictions))

              precision    recall  f1-score   support

         LOC       0.75      0.85      0.79       272
        MISC       0.73      0.56      0.63       133
         ORG       0.64      0.76      0.70       203
         PER       0.97      0.98      0.98       284

   micro avg       0.79      0.83      0.81       892
   macro avg       0.77      0.79      0.77       892
weighted avg       0.79      0.83      0.81       892



In [ ]:
# Inference

In [ ]:
sentence = "John works at Google in California"

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, prediction in zip(tokens, predictions[0].numpy()):
    print(token, label_names[prediction])

[CLS] O
john B-PER
works O
at O
google B-ORG
in O
california B-LOC
[SEP] O


In [ ]:
# Inference Analysis

In [ ]:
The trained DistilBERT model successfully identified named entities from the custom input sentence.

The model correctly recognized:
- John as a Person (PER)
- Google as an Organization (ORG)
- California as a Location (LOC)

This demonstrates that the transformer model effectively learned contextual token classification and sequence labeling patterns from the dataset.

In [ ]:
# Final Results

In [ ]:
Precision: 78.51%

Recall: 82.73%

F1 Score: 80.56%

In [ ]:
# Challenges Faced
- Handling subword tokenization
- Aligning labels with tokens
- Managing special tokens using -100
- Training large transformer models efficiently
- Understanding sequence labeling outputs

In [ ]:
# Observations and Insights
- DistilBERT performs well for token classification tasks.
- Proper label alignment is essential for accurate predictions.
- Transformer models capture contextual information effectively.
- Sequence labeling tasks require careful preprocessing.
- Fine-tuning improves model understanding of linguistic patterns.

In [ ]:
# Conclusion

- This project demonstrates the effectiveness of transformer-based models like DistilBERT for token classification tasks such as POS tagging and chunking.

- The model achieved strong sequence labeling performance and successfully identified grammatical and phrase-level structures in text data.

- Overall, the project highlights the importance of transformer models in modern Natural Language Processing applications.

In [25]:
from google.colab import files

uploaded = files.upload()

Saving Task5_POS_Tagging_Chunking.ipynb to Task5_POS_Tagging_Chunking.ipynb


In [26]:
import nbformat

notebook_path = "Task5_POS_Tagging_Chunking.ipynb"

nb = nbformat.read(notebook_path, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

fixed_path = "Fixed_Task5_Notebook.ipynb"

nbformat.write(nb, fixed_path)

print("Notebook fixed successfully!")

Notebook fixed successfully!
